<h1>🖥️ ColabDesk</h1>
<p><strong>A Free Google Colab Virtual Desktop (XFCE4) via Browser</strong></p>

---
### ⚡ Features
- **One-Click Run**: Everything is integrated into a single execution cell.
- **No App Needed**: Access the desktop directly from your web browser (noVNC).
- **Automated Setup**: Desktop environment and tunneling tools are installed automatically.

In [ ]:
# @title 🖥️ ColabDesk — Free Edition
# @markdown Fill in the options below and click the **Play** button on the left.
# @markdown ---
vnc_password = "colabdesk" # @param {type:"string"}
resolution = "1280x720" # @param ["1024x768", "1280x720", "1920x1080"]
# @markdown ---
# @markdown **Ngrok Token:** Paste your token from [ngrok.com](https://dashboard.ngrok.com).
ngrok_token = "" # @param {type:"string"}
# @markdown ---

import os
import time
import json
import urllib.request
from IPython.display import clear_output

# === Configuration ===
os.environ["DEBIAN_FRONTEND"] = "noninteractive"

# === 1. Cleanup ===
print("⏳ 1/4: Cleaning up previous sessions...")
os.system("vncserver -kill :1 > /dev/null 2>&1")
os.system("rm -rf /tmp/.X1* /tmp/.X11-unix")
os.system("pkill -f websockify")
os.system("pkill -f ngrok")

# === 2. Install Desktop (no extra apps) ===
if not os.path.exists('/usr/bin/vncserver'):
    print("⏳ 2/4: Installing XFCE4 Desktop & Dependencies (Takes ~2 minutes, please wait)...")
    os.system("apt-get update -qq > /tmp/apt.log 2>&1")
    os.system("apt-get install -y -qq xfce4 xfce4-goodies tightvncserver novnc websockify >> /tmp/apt.log 2>&1")
    os.system("apt-get remove -y -qq xfce4-power-manager xfce4-power-manager-plugins light-locker >> /tmp/apt.log 2>&1")
    os.system("sed -i '/power-manager-plugin/d' /etc/xdg/xfce4/panel/default.xml > /dev/null 2>&1")
else:
    print("✅ 2/4: Desktop Environment already installed, skipping...")

# === 3. Install Ngrok ===
if not os.path.exists('/usr/local/bin/ngrok'):
    print("⏳ 3/4: Downloading & Configuring Ngrok...")
    os.system("rm -f /usr/local/bin/ngrok")
    os.system("wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz")
    os.system("tar -xzf ngrok-v3-stable-linux-amd64.tgz -C /usr/local/bin > /dev/null 2>&1")
    os.system("rm ngrok-v3-stable-linux-amd64.tgz")
else:
    print("✅ 3/4: Ngrok already installed, skipping...")

# === 4. Setup & Start VNC ===
os.system("mkdir -p ~/.vnc")
os.system(f"echo '{vnc_password}' | vncpasswd -f > ~/.vnc/passwd")
os.system("chmod 600 ~/.vnc/passwd")

xstartup = """#!/bin/bash
xrdb $HOME/.Xresources
startxfce4 &
"""
with open("/root/.vnc/xstartup", "w") as f:
    f.write(xstartup)
os.system("chmod +x /root/.vnc/xstartup")

print(f"⏳ 4/4: Starting VNC Server ({resolution}) & noVNC...")
os.system(f"USER=root vncserver -geometry {resolution} :1 > /dev/null 2>&1")
os.system("websockify --web /usr/share/novnc 6080 localhost:5901 > /dev/null 2>&1 &")

# === 5. Start Ngrok (single token, no fallback) ===
if not ngrok_token.strip():
    clear_output()
    print("❌ ERROR: You must provide an Ngrok token in the 'ngrok_token' field!")
else:
    os.system(f"ngrok config add-authtoken {ngrok_token.strip()} > /dev/null 2>&1")
    os.system("ngrok http 6080 --log=stdout > /tmp/ngrok.log 2>&1 &")
    time.sleep(5)

    ngrok_url = None
    try:
        req = urllib.request.Request("http://localhost:4040/api/tunnels")
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
            if data.get('tunnels'):
                ngrok_url = data['tunnels'][0]['public_url']
    except Exception:
        pass

    clear_output()

    if ngrok_url:
        print("🎉 COLABDESK IS READY! 🎉")
        print("=" * 60)
        print(f"🔗 Open your desktop: {ngrok_url}/vnc.html")
        print("=" * 60)
        print(f"👉 VNC Password: {vnc_password}")
        print("\n⚠️ IMPORTANT: Keep this Colab tab open so the session doesn't disconnect.")
        print("\n" + "=" * 60)
        print("⭐ UPGRADE TO COLABDESK PRO")
        print("=" * 60)
        print("Unlock premium features:")
        print("  ✅ Smart Multi-Token Fallback (auto-retry on rate limits)")
        print("  ✅ Rate-Limit Detection & Token Rotation")
        print("  ✅ Google Chrome Pre-installed & Auto-started")
        print("  ✅ Auto-Connect VNC (no manual password entry)")
        print("  ✅ Auto-Reconnect with configurable delay")
        print("👉 Get it here: https://lynk.id/achmadhadikurnia")
    else:
        print("❌ FAILED: Could not establish an Ngrok tunnel.")
        print("Please check your token and try again.")
        print("Get a token from: https://dashboard.ngrok.com")